<a href="https://colab.research.google.com/github/RautRitesh/slm_project/blob/main/Ritesh_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
token=userdata.get('HF_TOKEN')

In [ ]:
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from dataclasses import dataclass
from typing import Optional, List
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from peft import PeftModel
from trl import SFTTrainer, SFTConfig

@dataclass
class FineTuneConfig:
    # model
    model_name: str = "unsloth/Qwen2-1.5B-Instruct"
    load_in_4bit: bool = True
    max_seq_length: int = 1024 # Safe for Colab T4
    dtype: Optional[str] = None

    # dataset
    dataset_name: str = "zeri000/augmented_nepali_legal_qa.csv"
    split: str = "train"
    seed: int = 3407

    # training
    output_dir: str = "nepali_legal_checkpoints"
    lora_save_path: str = "nepali_legal_lora"
    per_device_bs: int = 2
    grad_acc_steps: int = 4
    epochs: int = 3
    lr: float = 2e-4
    warmup_ratio: float = 0.1
    logging_steps: int = 10
    packing: bool = False

    # lora
    lora_r: int = 16
    lora_alpha: int = 32 # 🟢 FIXED: Alpha must be 2x Rank for the model to learn effectively
    lora_dropout: float = 0.0
    target_modules: List[str] = None
    use_gc: bool = False

    # save merged model
    save_merged: bool = False
    merged_save_path: str = "merged_model"

class UnslothFineTuner:
    def __init__(self, cfg: FineTuneConfig):
        self.cfg = cfg
        if self.cfg.target_modules is None:
            self.cfg.target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                                     "gate_proj", "up_proj", "down_proj"]
        self.model = None
        self.tokenizer = None

    # ---------- Load Model ----------
    def load_model(self):
        print(f"✅ Loading model: {self.cfg.model_name}")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.model_name,
            max_seq_length=self.cfg.max_seq_length,
            dtype=self.cfg.dtype,
            load_in_4bit=self.cfg.load_in_4bit,
        )

        # Apply ChatML Template for Qwen Instruct models
        self.tokenizer = get_chat_template(
            self.tokenizer,
            chat_template = "chatml",
        )

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.cfg.lora_r,
            target_modules=self.cfg.target_modules,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=True,
            random_state=self.cfg.seed,
        )
        print("✅ Model loaded successfully.")

    # ---------- Load Dataset ----------
    def load_dataset(self):
        print(f"✅ Loading dataset: {self.cfg.dataset_name}")
        ds = load_dataset(self.cfg.dataset_name, split=self.cfg.split)
        ds = ds.shuffle(seed=self.cfg.seed)
        return ds

    # ---------- Formatting (Native ChatML) ----------
    def _format_nepali_legal(self, batch):
        texts = []
        inp_list = batch.get("input", [""] * len(batch["instruction"]))

        for ins, inp, out in zip(batch["instruction"], inp_list, batch["output"]):
            user_text = ins
            if inp:
                user_text += f"\n\nसन्दर्भ (Context):\n{inp}"

            # Create a ChatML structured conversation
            messages = [
                {"role": "system", "content": "तपाईं एक विशेषज्ञ नेपाली कानूनी सहायक हुनुहुन्छ।"},
                {"role": "user", "content": user_text},
                {"role": "assistant", "content": out}
            ]

            # Apply the native template
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)

        return {"text": texts}

    # ---------- Train ----------
    def train(self, ds):
        print("✅ Starting training...")
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=ds,
            dataset_text_field="text",
            max_seq_length=self.cfg.max_seq_length,
            packing=self.cfg.packing,
            args=SFTConfig(
                per_device_train_batch_size=self.cfg.per_device_bs,
                gradient_accumulation_steps=self.cfg.grad_acc_steps,
                num_train_epochs=self.cfg.epochs,
                learning_rate=self.cfg.lr,
                warmup_ratio=self.cfg.warmup_ratio,
                fp16 = not torch.cuda.is_bf16_supported(),
                bf16 = torch.cuda.is_bf16_supported(),
                logging_steps=self.cfg.logging_steps,
                output_dir=self.cfg.output_dir,
                seed=self.cfg.seed,
            ),
        )
        trainer.train()

    # ---------- Save ----------
    def save(self):
        print(f"✅ Saving adapters to: {self.cfg.lora_save_path}")
        self.model.save_pretrained(self.cfg.lora_save_path)
        self.tokenizer.save_pretrained(self.cfg.lora_save_path)

        if self.cfg.save_merged:
            print(f"✅ Saving merged model to: {self.cfg.merged_save_path}")
            self.model.save_pretrained_merged(self.cfg.merged_save_path, self.tokenizer, save_method="merged_16bit")

    # ---------- Run ----------
    def run(self):
        self.load_model()
        raw_ds = self.load_dataset()

        if "instruction" in raw_ds.column_names and "output" in raw_ds.column_names:
            print("✅ Detected instruction/output format. Applying ChatML.")
            ds = raw_ds.map(self._format_nepali_legal, batched=True)
        else:
            raise ValueError(f"Dataset columns {raw_ds.column_names} do not match expected format.")

        self.train(ds)
        self.save()
        print("✅ Fine-tuning Complete!")

cfg = FineTuneConfig()
trainer = UnslothFineTuner(cfg)
trainer.run()

FastLanguageModel.for_inference(trainer.model)

# 🟢 FIXED: Create stopping criteria for ChatML so it knows when to stop generating
terminators = [
    trainer.tokenizer.eos_token_id,
    trainer.tokenizer.convert_tokens_to_ids("<|im_end|>")
]

def test_inference(input_text):
    messages = [
        {"role": "system", "content": "तपाईं एक विशेषज्ञ नेपाली कानूनी सहायक हुनुहुन्छ।"},
        {"role": "user", "content": input_text}
    ]
    prompt = trainer.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = trainer.tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=512,           # Increased to allow full answers
        use_cache=True,
        temperature=0.01,             # 🟢 FIXED: Ultra-low temp for factual legal recall
        top_p=0.95,
        repetition_penalty=1.0,       # 🟢 FIXED: Disabled penalty to protect Nepali grammar
        eos_token_id=terminators,     # 🟢 FIXED: Stops generation naturally
        do_sample=True
    )

    # Calculate the length of the input prompt to slice it out of the output
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]

    # Decode only the newly generated text
    response = trainer.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return response

print("\n=== MODEL ANSWER 1 ===\n")
ans1 = test_inference("मुलुकी अपराध संहिता?")
print(ans1)

print("\n=== MODEL ANSWER 2 ===\n")
ans2 = test_inference("सम्बन्ध विच्छेद कसरी प्राप्त गर्ने?")
print(ans2)

# ==========================================
# 3) PUSH TO HUB
# ==========================================
# Define your repository name (Update the '4' to differentiate from your previous run)
repo_name = "zeri000/nepali_legal_qwen_merged_4"

# Merge and Push (16bit is recommended for best quality/size balance)
trainer.model.push_to_hub_merged(
    repo_name,
    trainer.tokenizer,
    save_method = "merged_16bit",
    # token = "hf_your_token_here" # Ensure you have logged in via huggingface-cli login
)

print(f"✅ Full merged model pushed to: https://huggingface.co/{repo_name}")

In [4]:
print("\n=== MODEL ANSWER 1 ===\n")
ans1 = test_inference("मुलुकी अपराध संहिता?")
print(ans1)

print("\n=== MODEL ANSWER 2 ===\n")
ans2 = test_inference("सम्बन्ध विच्छेद कसरी प्राप्त गर्ने?")
print(ans2)


=== MODEL ANSWER 1 ===

यहाँको प्रश्नको सन्दर्भमा, मुलुकी अपराध संहिता भनेको नेपालको अपराध ऐन, २००६ लाई अनुमोदन गरी नेपाल सरकारले बनाएको ऐन हो।

=== MODEL ANSWER 2 ===

नेपालको कानुन अनुसार, सम्बन्ध विच्छेद गर्ने प्रक्रियामा दुवै पति र पत्नीको सहमतिमा र यदि उनीहरू बसोबास छन् भने, उनीहरूको बसोबासको ठेगानामा सम्बन्ध विच्छेद दर्ता प्रमाणपत्र जारी गरिन्छ।


In [ ]:
repo_name = "zeri000/nepali_legal_qwen_merged_4"

# Merge and Push (16bit is recommended for best quality/size balance)
trainer.model.push_to_hub_merged(
    repo_name,
    trainer.tokenizer,
    save_method = "merged_16bit",
    token = token # Ensure you have logged in via huggingface-cli login
)

print(f"✅ Full merged model pushed to: https://huggingface.co/{repo_name}")

Testing the model

In [7]:
print("\n=== MODEL ANSWER 1 ===\n")
ans1 = test_inference("श्रीमान् वा श्रीमतीमध्ये कसले सम्बन्ध विच्छेदको मुद्दा हाल्न सक्छ??")
print(ans1)

print("\n=== MODEL ANSWER 2 ===\n")
ans2 = test_inference("चोरीको मुद्दामा के कस्तो सजाय हुन्छ?")
print(ans2)


=== MODEL ANSWER 1 ===

प्रचलित कानुनको आधारमा भन्नुपर्दा, जस्तो भएमा श्रीमान् वा श्रीमतीमध्ये कुनै एकले आफ्नो विरुद्धमा सम्बन्ध विच्छेदको मुद्दा दर्ता गर्न सक्छ।

=== MODEL ANSWER 2 ===

प्रचलित कानुनको आधारमा भन्नुपर्दा, दफा १४५ बमोजिम चोरी गरेमा वा गराउनेलाई दुई वर्षसम्म कैद वा दुई हजार रुपैयाँसम्म जरिवाना वा दुवै सजाय हुनेछ।
